In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

print(f"Source System: {source_system}")
print(f"Table Name: {table_name}")
print(f"Landing Timestamp: {landing_timestamp}")
print(f"Source File Format: {source_file_format}")

In [0]:
# Construct the landing path
landing_path = f"/Volumes/adb_caresynce/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"
print(f"Landing path: {landing_path}")

# Check if the path exists; exit gracefully if not
try:
    files = dbutils.fs.ls(landing_path)
    if not files:
        print("No new files received at the landing path. Exiting notebook.")
        dbutils.notebook.exit("No new files received")
    print(f"Found {len(files)} file(s) at landing path.")
except Exception:
    print(f"Path does not exist: {landing_path}")
    print("No new files received. Exiting notebook.")
    dbutils.notebook.exit("No new files received - path not found")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresynce.bronze.{table_name}")